In [2]:
import pickle
import random
import numpy as np
import pandas as pd
from tqdm import tqdm
from datetime import datetime
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [3]:
df = pd.read_csv(f'../processed_data/final_df.csv')

In [4]:
# Read the pathway mask
pathway_mask = np.load(f'../processed_data/pathway_mask.npy')
print(f"pathway_mask.shape: {pathway_mask.shape}")

pathway_mask.shape: (231, 6353)


In [12]:
random_seeds1 = random.sample(range(1, 101), 10)

with open('random_seeds1.pickle', 'wb') as file:
    pickle.dump(random_seeds1, file)

In [13]:
random_seeds2 = random.sample(range(101, 201), 10)

with open('random_seeds2.pickle', 'wb') as file:
    pickle.dump(random_seeds2, file)

In [14]:
df.head()

,A2M,AAAS,AACS,AADAT,AANAT,AARS2,AASDHPPT,AASS,ABAT,ABCA10,...,ZNF274,ZNF385A,ZNF516,ZNF76,ZNFX1,ZNHIT1,ZNRF3,ZYX,OS_MONTHS,OS_STATUS
0,-0.5741,0.8255,-0.3945,-0.5373,1.7946,0.1707,-0.4719,-1.0813,-0.8503,0.3254,...,-1.5938,1.2525,-1.8470,1.2804,0.3560,1.2492,0.1645,0.9335,131.57,0
1,-0.3511,0.1441,0.9249,0.1542,1.4301,-0.0218,-0.1214,-1.0040,-0.5136,0.6606,...,0.2706,0.9741,-1.2112,-0.0348,-1.1744,1.1303,-0.4431,0.9163,48.42,0
2,-0.0415,0.3814,-0.3386,0.2124,-0.6428,-0.0695,-0.5220,0.1486,0.6390,0.5450,...,0.3481,0.7387,0.4739,0.5981,-0.8679,0.4004,-0.0372,1.0020,47.57,0
3,-0.2346,0.5571,-0.3850,0.3001,-0.6177,0.7093,-1.4252,1.1307,0.1071,0.7576,...,-0.0478,0.0584,0.4055,0.3299,-0.4042,0.0366,0.0807,0.2990,11.43,0
4,0.1282,1.0654,-0.4018,-0.2326,-1.6857,0.7334,-0.6222,1.4037,0.1067,0.7381,...,0.7434,-0.3379,-0.9181,0.4533,-0.6451,0.3672,-0.4134,0.3444,48.52,0


In [15]:
gene_columns = [col for col in df.columns if col not in ["PATIENT_ID", "CANCER_TYPE", "SAMPLE_TYPE_ID", "VIAL_NUMBER", "OS_MONTHS", "OS_STATUS"]]

In [1]:
for i, (seed1, seed2) in enumerate(zip(random_seeds1, random_seeds2)):
    
    print(f"#######################  {i+1} experiment  #######################\n")
    
    all_indices = pd.DataFrame({"indices": [j for j in range(len(df))]})
    y = df[['OS_MONTHS', 'OS_STATUS']]
    
    # Train and Test Split
    train_indices, test_indices, y_train, y_test = train_test_split(all_indices, y, test_size=0.2, stratify=y["OS_STATUS"], random_state=seed1)
    
    # Val and Test Split
    val_indices, test_indices, y_val, y_test = train_test_split(test_indices, y_test, test_size=0.5, stratify=y_test["OS_STATUS"], random_state=seed2)
    
    # Saving the indices
    train_indices.to_csv(f'data_splits/{i+1}/exp_{i+1}_indices_train.csv', index=False)
    val_indices.to_csv(f'data_splits/{i+1}/exp_{i+1}_indices_val.csv', index=False)
    test_indices.to_csv(f'data_splits/{i+1}/exp_{i+1}_indices_test.csv', index=False)
    
    # Separate Data
    X_train = df.loc[train_indices["indices"].values, df.columns].drop(columns=["OS_MONTHS", "OS_STATUS"]).values
    X_val   = df.loc[val_indices["indices"].values, df.columns].drop(columns=["OS_MONTHS", "OS_STATUS"]).values
    X_test  = df.loc[test_indices["indices"].values, df.columns].drop(columns=["OS_MONTHS", "OS_STATUS"]).values
    
    data_scaler = StandardScaler()
    data_scaler.fit(X_train)
    
    X_train_scaled = data_scaler.transform(X_train)
    X_val_scaled   = data_scaler.transform(X_val)
    X_test_scaled  = data_scaler.transform(X_test)
  
    with open(f'data_splits/{i+1}/data_scaler.pickle', 'wb') as file:
        pickle.dump(data_scaler, file)
    
    np.save(f'data_splits/{i+1}/X_train_{i+1}.npy', X_train_scaled)
    np.save(f'data_splits/{i+1}/X_val_{i+1}.npy', X_val_scaled)
    np.save(f'data_splits/{i+1}/X_test_{i+1}.npy', X_test_scaled)
    
    np.save(f'data_splits/{i+1}/y_train_{i+1}.npy', y_train)
    np.save(f'data_splits/{i+1}/y_val_{i+1}.npy', y_val)
    np.save(f'data_splits/{i+1}/y_test_{i+1}.npy', y_test)
    
    print(f'exp_num: {i+1}')
    print(f'X_train.shape: {X_train.shape}, y_train.shape: {y_train.shape}')
    print(f'X_val.shape: {X_val.shape}, y_val.shape: {y_val.shape}')
    print(f'X_test.shape: {X_test.shape}, y_test.shape: {y_test.shape}')

###### 

In [19]:
y_test = np.load(f'data_splits/{i+1}/y_test_{i+1}.npy')

In [6]:
for EXP_NUM in tqdm(range(10)):
    
    X_train = np.load(f'data_splits/{EXP_NUM+1}/X_train_{EXP_NUM+1}.npy')
    X_val = np.load(f'data_splits/{EXP_NUM+1}/X_val_{EXP_NUM+1}.npy')
    X_test = np.load(f'data_splits/{EXP_NUM+1}/X_test_{EXP_NUM+1}.npy')
    
    X_train_3d = np.asarray([X_train[i]*pathway_mask for i in range(len(X_train))])
    X_val_3d = np.asarray([X_val[i]*pathway_mask for i in range(len(X_val))])
    X_test_3d = np.asarray([X_test[i]*pathway_mask for i in range(len(X_test))])
    
    np.save(f'data_splits/{EXP_NUM+1}/X_train_3d_{EXP_NUM+1}.npy', X_train_3d)
    np.save(f'data_splits/{EXP_NUM+1}/X_val_3d_{EXP_NUM+1}.npy', X_val_3d)
    np.save(f'data_splits/{EXP_NUM+1}/X_test_3d_{EXP_NUM+1}.npy', X_test_3d)

100%|██████████| 20/20 [09:53<00:00, 29.70s/it]


In [7]:
# np.unique(y_train['OS_STATUS'], return_counts=True)

In [19]:
np.unique(y_test['OS_STATUS'], return_counts=True)

(array([0, 1]), array([38, 13]))

In [20]:
np.unique(y_val['OS_STATUS'], return_counts=True)

(array([0, 1]), array([38, 12]))